# Finetune MANNER (thay cho FullSubNet)
Notebook nay duoc chuyen doi tu file FullSubNet finetune cu sang mo hinh **MANNER** (https://github.com/winddori2002/MANNER), su dung checkpoint pretrain tai tu Google Drive, va co them **Early Stopping** (dung som neu PESQ tren validation khong cai thien, toi da 20 epoch).

In [ ]:
!nvidia-smi


In [ ]:
%cd /kaggle/working
!rm -rf MANNER
!git clone https://github.com/winddori2002/MANNER.git
%cd MANNER

# Go sach numpy/scipy roi cai lai dung 1 cap phien ban da biet tuong thich voi nhau,
# thay vi de pip tu chon ban moi nhat (de gay xung dot ABI giua cac ban .so bien dich san).
!pip uninstall -y -q numpy scipy
!pip install -q --no-cache-dir --force-reinstall numpy==1.26.4 scipy==1.11.4
# --no-deps de tranh cac goi ben duoi keo theo nang cap lai numpy/scipy them lan nua
!pip install -q --no-deps -U librosa soundfile pystoi tqdm neptune-client gdown toml
# pesq: KHONG dung wheel dung san (build cho numpy2, ABI size 96) vi runtime la numpy 1.26.4 (ABI size 88)
# --no-binary pesq bat pip build lai tu source, khop voi numpy dang cai => het loi 'dtype size changed'
!pip uninstall -y -q pesq
!pip install -q --no-cache-dir -U cython setuptools wheel  # can thiet vi --no-build-isolation khong tu cai ho
# --no-build-isolation: KHONG cho pip tao moi truong build rieng (moi truong do se tu tai numpy moi nhat
# de build pesq, gay lech ABI y het loi cu). Bat no dung thang numpy 1.26.4 da cai san de build.
!pip install -q --no-cache-dir --no-binary pesq --no-build-isolation --no-deps pesq

# QUAN TRONG: sau cell nay PHAI Restart kernel (Run > Restart & Run All) truoc khi chay tiep,
# vi numpy/scipy da nap vao bo nho tu dau session, pip install/uninstall moi khong co tac dung neu khong restart.


In [ ]:
import torch
print("torch version:", torch.__version__)
print("torch cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Torch dang khong thay CUDA - restart session roi chay lai tu Buoc 0"


In [ ]:
# MANNER dung torch.stft() kieu cu (tra ve tensor [..., 2] cho real/imag).
# Cac ban PyTorch moi (>=2.1) bat buoc phai truyen return_complex=True, nen can patch lai
# ham stft() trong src/stft_loss.py de tranh loi RuntimeError khi tinh STFT loss.
stft_file = "/kaggle/working/MANNER/src/stft_loss.py"
with open(stft_file, "r") as f:
    src = f.read()

new_src = src.replace(
    "x_stft = torch.stft(x, fft_size, hop_size, win_length, window)",
    "x_stft = torch.stft(x, fft_size, hop_size, win_length, window, return_complex=True)"
)
new_src = new_src.replace(
    "real   = x_stft[..., 0]",
    "real   = x_stft.real"
)
new_src = new_src.replace(
    "imag   = x_stft[..., 1]",
    "imag   = x_stft.imag"
)

if new_src != src:
    with open(stft_file, "w") as f:
        f.write(new_src)
    print("Da patch xong:", stft_file)
else:
    print("Khong tim thay doan can patch (co the da patch roi hoac code goc da doi khac).")

import py_compile
py_compile.compile(stft_file, doraise=True)
print("OK: stft_loss.py hop le ve cu phap sau khi patch.")


In [ ]:
# MANNER import "neptune" o muc module (trong src/utils.py) chi de logging thi nghiem,
# nhung ta khong dung tinh nang nay va cai dat neptune tren Kaggle rat de bi gay
# (thieu bravado_core, xung dot version...). Vi vay patch lai de import neptune la TUY CHON,
# tranh ca chuoi import bi crash chi vi 1 thu vien logging khong dung toi.
utils_file = "/kaggle/working/MANNER/src/utils.py"
with open(utils_file, "r") as f:
    src = f.read()

new_src = src.replace(
    "import neptune\n",
    "try:\n    import neptune\nexcept Exception:\n    neptune = None  # khong bat buoc, chi dung khi logging=True\n"
)

if new_src != src:
    with open(utils_file, "w") as f:
        f.write(new_src)
    print("Da patch xong:", utils_file)
else:
    print("Khong tim thay dong can patch (co the da patch roi).")

import py_compile
py_compile.compile(utils_file, doraise=True)
print("OK: utils.py hop le ve cu phap sau khi patch.")


In [ ]:
import os

CLEAN_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
NOISE_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"
TEST_DIR  = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST "

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isdir(CLEAN_DIR), f"Khong tim thay: {CLEAN_DIR}"
assert os.path.isdir(NOISE_DIR), f"Khong tim thay: {NOISE_DIR}"
assert os.path.isdir(TEST_DIR), f"Khong tim thay: {TEST_DIR}"
print("OK, cac thu muc du lieu ton tai.")


In [ ]:
AUDIO_EXT = (".wav", ".flac")

def list_audio_files(folder, exts=AUDIO_EXT):
    files = []
    for root, _, names in os.walk(folder):
        for n in names:
            if n.lower().endswith(exts):
                files.append(os.path.join(root, n))
    return sorted(files)

clean_files = list_audio_files(CLEAN_DIR)
noise_files = list_audio_files(NOISE_DIR)
test_files  = list_audio_files(TEST_DIR)

print("So file clean:", len(clean_files))
print("So file noise:", len(noise_files))
print("So file test (noisy):", len(test_files))

assert len(clean_files) > 0, "Khong tim thay file audio nao trong CLEAN_DIR"
assert len(noise_files) > 0, "Khong tim thay file audio nao trong NOISE_DIR"


In [ ]:
import random, numpy as np, soundfile as sf

random.seed(0)
TARGET_SR = 16000

def load_mono(path, target_sr=TARGET_SR):
    wav, sr = sf.read(path, always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        import librosa
        wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=target_sr)
    return wav.astype(np.float32)

def mix_snr(clean_wav, noise_wav, snr_db=10):
    if len(noise_wav) < len(clean_wav):
        reps = len(clean_wav) // len(noise_wav) + 1
        noise_wav = np.tile(noise_wav, reps)
    noise_wav = noise_wav[: len(clean_wav)]
    clean_power = np.mean(clean_wav ** 2) + 1e-8
    noise_power = np.mean(noise_wav ** 2) + 1e-8
    scale = np.sqrt(clean_power / (noise_power * (10 ** (snr_db / 10))))
    noisy = clean_wav + noise_wav * scale
    return noisy

# --- Tao bo validation co dinh (mix san 1 lan, dung de danh gia PESQ moi epoch) ---
N_VAL = min(40, len(clean_files) // 20 + 1)
val_clean_pool = random.sample(clean_files, min(N_VAL, len(clean_files)))
val_noise_pool = [random.choice(noise_files) for _ in val_clean_pool]

VAL_DIR = os.path.join(WORK_DIR, "manner_val")
VAL_CLEAN_DIR = os.path.join(VAL_DIR, "clean")
VAL_NOISY_DIR = os.path.join(VAL_DIR, "noisy")
os.makedirs(VAL_CLEAN_DIR, exist_ok=True)
os.makedirs(VAL_NOISY_DIR, exist_ok=True)

for i, (c_path, n_path) in enumerate(zip(val_clean_pool, val_noise_pool)):
    clean_wav = load_mono(c_path)
    noise_wav = load_mono(n_path)
    snr_db = random.uniform(0, 15)
    noisy_wav = mix_snr(clean_wav, noise_wav, snr_db)
    sf.write(os.path.join(VAL_CLEAN_DIR, f"clean_fileid_{i}.wav"), clean_wav, TARGET_SR)
    sf.write(os.path.join(VAL_NOISY_DIR, f"noisy_fileid_{i}.wav"), noisy_wav, TARGET_SR)

print(f"Da tao {len(val_clean_pool)} cap validation tai {VAL_DIR}")

val_clean_set = set(val_clean_pool)
clean_files_train = [f for f in clean_files if f not in val_clean_set]
noise_files_train = noise_files

print("So file clean con lai cho train:", len(clean_files_train))


In [ ]:
# Tai checkpoint pretrain cua MANNER tu Google Drive
# Link goc: https://drive.google.com/file/d/1_JYVnqwwuyziq4nLK8QSAomOaWHVIr-z/view?usp=sharing
import gdown

FILE_ID  = "1_JYVnqwwuyziq4nLK8QSAomOaWHVIr-z"
CKPT_PATH = "/kaggle/working/manner_pretrained.pth"

gdown.download(id=FILE_ID, output=CKPT_PATH, quiet=False)

size_mb = os.path.getsize(CKPT_PATH) / (1024 * 1024)
print(f"Da tai checkpoint ve: {CKPT_PATH} ({size_mb:.1f} MB)")
assert size_mb > 1, ("File qua nho, co the Google Drive tra ve trang HTML loi thay vi file that. "
                      "Kiem tra lai quyen chia se cua file (phai la 'Anyone with the link').")


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/MANNER")

import torch
from src.models import MANNER as MANNER_BASE
# Neu checkpoint ban tai la ban 'small', doi sang dong duoi:
# from src.models_small import MANNER as MANNER_BASE

MODEL_ARGS = dict(
    in_channels=1,
    out_channels=1,
    hidden=60,        # doi thanh 120 neu checkpoint la ban 'large'
    depth=4,
    kernel_size=8,
    stride=4,
    growth=2,
    head=1,
    segment_len=64,
)

model = MANNER_BASE(**MODEL_ARGS)

ckpt_raw = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
if isinstance(ckpt_raw, dict) and "state_dict" in ckpt_raw:
    state_dict = ckpt_raw["state_dict"]
elif isinstance(ckpt_raw, dict) and "model" in ckpt_raw:
    state_dict = ckpt_raw["model"]
else:
    state_dict = ckpt_raw

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
print("Neu 2 danh sach tren rong -> checkpoint khop hoan toan voi kien truc MANNER (MODEL_ARGS dung).")
print("Neu khong rong -> kiem tra lai hidden (60/120) hoac dung ban 'small' truoc khi finetune tiep.")


In [ ]:
CLEAN_CKPT_PATH = os.path.join(WORK_DIR, "manner_pretrained_clean.pth")
torch.save({"epoch": 0, "state_dict": model.state_dict()}, CLEAN_CKPT_PATH)
print("Da luu checkpoint sach tai:", CLEAN_CKPT_PATH)


In [ ]:
import random
from torch.utils.data import Dataset, DataLoader

SEGMENT_SEC = 4
SEGMENT_LEN = SEGMENT_SEC * TARGET_SR   # 64000 mau tai 16kHz, giong config mac dinh cua MANNER
SNR_RANGE   = (0, 15)                   # cung khoang SNR nhu khi tao validation set

class OnTheFlyNoisyDataset(Dataset):
    """
    Voi moi lan lay mau: crop ngau nhien 1 doan clean_speech, chon ngau nhien 1 file noise,
    mix o SNR ngau nhien -> tao cap (noisy, clean) moi moi epoch (data augmentation).
    Thay the cho kieu 'clean.txt/noise.txt' cua FullSubNet, vi MANNER can 1 Dataset tra ve
    truc tiep cap (noisy, clean) cung do dai.
    """
    def __init__(self, clean_files, noise_files, segment_len=SEGMENT_LEN, snr_range=SNR_RANGE, sr=TARGET_SR):
        self.clean_files = clean_files
        self.noise_files = noise_files
        self.segment_len = segment_len
        self.snr_range   = snr_range
        self.sr          = sr

    def __len__(self):
        return len(self.clean_files)

    def _crop_or_pad(self, wav):
        if len(wav) >= self.segment_len:
            start = random.randint(0, len(wav) - self.segment_len)
            return wav[start:start + self.segment_len]
        return np.pad(wav, (0, self.segment_len - len(wav)))

    def __getitem__(self, idx):
        clean_wav = self._crop_or_pad(load_mono(self.clean_files[idx], target_sr=self.sr))

        noise_wav = load_mono(random.choice(self.noise_files), target_sr=self.sr)
        if len(noise_wav) < self.segment_len:
            reps = self.segment_len // len(noise_wav) + 1
            noise_wav = np.tile(noise_wav, reps)
        start = random.randint(0, len(noise_wav) - self.segment_len)
        noise_wav = noise_wav[start:start + self.segment_len]

        noisy_wav = mix_snr(clean_wav, noise_wav, random.uniform(*self.snr_range))

        noisy = torch.from_numpy(noisy_wav).float().unsqueeze(0)
        clean = torch.from_numpy(clean_wav).float().unsqueeze(0)
        return noisy, clean, 0, 0


class PairedFileDataset(Dataset):
    """Doc cap file (noisy, clean) da mix san tren dia - dung cho validation co dinh."""
    def __init__(self, clean_dir, noisy_dir):
        self.clean_files = list_audio_files(clean_dir)
        self.noisy_files = list_audio_files(noisy_dir)
        assert len(self.clean_files) == len(self.noisy_files), "So file clean/noisy validation khong khop"

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        clean_wav, _ = sf.read(self.clean_files[idx], dtype="float32")
        noisy_wav, _ = sf.read(self.noisy_files[idx], dtype="float32")
        clean = torch.from_numpy(clean_wav).float().unsqueeze(0)
        noisy = torch.from_numpy(noisy_wav).float().unsqueeze(0)
        return noisy, clean, 0, 0


BATCH_SIZE = 4   # giam neu bi OOM tren GPU Kaggle

train_dataset = OnTheFlyNoisyDataset(clean_files_train, noise_files_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

val_dataset = PairedFileDataset(VAL_CLEAN_DIR, VAL_NOISY_DIR)
val_loader  = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

print(f"So mau train moi epoch: {len(train_dataset)} | So mau validation: {len(val_dataset)}")


In [ ]:
from tqdm import tqdm
import numpy as np
if not hasattr(np, "complex"):
    np.complex = complex  # fix cho librosa cu / cache neu con dung alias np.complex da bi xoa
from src.time_loss import L1Loss, WeightedLoss
from src.stft_loss import MultiResolutionSTFTLoss
from src.metric import get_scores

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(DEVICE)

criterion = L1Loss().to(DEVICE)                 # doi sang L1CharbonnierLoss neu muon dung 'ch' loss
stft_loss = MultiResolutionSTFTLoss(factor_sc=0.5, factor_mag=0.5).to(DEVICE)

class _ScoringArgs:
    sample_rate = TARGET_SR
scoring_args = _ScoringArgs()

LEARNING_RATE = 1e-5   # tuong ung default cua MANNER khi finetune tu checkpoint co san
MAX_EPOCHS    = 20     # toi da 20 epoch nhu yeu cau
PATIENCE      = 5      # so epoch lien tiep khong cai thien PESQ truoc khi dung som

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3,
                                                 steps_per_epoch=len(train_loader), epochs=MAX_EPOCHS)


In [ ]:
SAVE_DIR = "/kaggle/working/Experiments/MANNER"
os.makedirs(SAVE_DIR, exist_ok=True)
BEST_CKPT_PATH = os.path.join(SAVE_DIR, "manner_finetuned_best.pth")
LAST_CKPT_PATH = os.path.join(SAVE_DIR, "manner_finetuned_last.pth")


def run_epoch(loader, train=True):
    total_loss, total_pesq, total_stoi, total_cnt = 0.0, 0.0, 0.0, 0
    model.train() if train else model.eval()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for i, (noisy, clean, _, _) in enumerate(tqdm(loader)):
            noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
            noise_label  = noisy - clean

            estimate       = model(noisy)
            noise_estimate = noisy - estimate

            loss       = criterion(clean.squeeze(1), estimate.squeeze(1))
            noise_loss = criterion(noise_label.squeeze(1), noise_estimate.squeeze(1))

            sc_loss, mag_loss = stft_loss(estimate.squeeze(1), clean.squeeze(1))
            loss += sc_loss + mag_loss
            sc_loss, mag_loss = stft_loss(noise_estimate.squeeze(1), noise_label.squeeze(1))
            noise_loss += sc_loss + mag_loss

            loss = WeightedLoss(clean.squeeze(1), noise_label.squeeze(1), loss, noise_loss)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()
            else:
                est_cpu, clean_cpu = estimate.cpu(), clean.cpu()
                total_cnt += clean_cpu.shape[0]
                p, s = get_scores(clean_cpu, est_cpu, scoring_args)
                total_pesq += p
                total_stoi += s

            total_loss += loss.item()

    if train:
        return total_loss / (i + 1)
    return total_loss / (i + 1), total_pesq / total_cnt, total_stoi / total_cnt


best_pesq         = -1.0
epochs_no_improve = 0
history           = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss, val_pesq, val_stoi = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | train loss: {train_loss:.4f} | "
          f"val loss: {val_loss:.4f} | PESQ: {val_pesq:.4f} | STOI: {val_stoi:.4f}")
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                         val_pesq=val_pesq, val_stoi=val_stoi))

    torch.save({"epoch": epoch, "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict()}, LAST_CKPT_PATH)

    if val_pesq > best_pesq:
        best_pesq = val_pesq
        epochs_no_improve = 0
        torch.save({"epoch": epoch, "loss": best_pesq, "state_dict": model.state_dict(),
                    "optimizer": optimizer.state_dict()}, BEST_CKPT_PATH)
        print(f"  -> PESQ cai thien, da luu checkpoint tot nhat tai {BEST_CKPT_PATH}")
    else:
        epochs_no_improve += 1
        print(f"  -> Khong cai thien ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping tai epoch {epoch}: PESQ khong cai thien sau {PATIENCE} epoch lien tiep.")
            break

print("Hoan tat finetune. PESQ tot nhat tren validation:", best_pesq)


## Suy luan (inference) tren tap test that su (chi co file noisy, khong co clean)
Vi `TEST_DIR` chi chua file nhieu (khong co ban sach de doi chieu), o buoc nay chi tang cuong am thanh va luu file `.wav` ket qua, giong cach `inference.py` cua FullSubNet hoac `custom_enhance.py` cua MANNER hoat dong (khong tinh PESQ/STOI o day).

In [ ]:
OUTPUT_DIR = "/kaggle/working/enhanced_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

infer_model = MANNER_BASE(**MODEL_ARGS).to(DEVICE)
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE, weights_only=False)
infer_model.load_state_dict(best_ckpt["state_dict"])
infer_model.eval()

print(f"Dang tang cuong {len(test_files)} file test, luu ket qua vao {OUTPUT_DIR} ...")
with torch.no_grad():
    for f_path in tqdm(test_files):
        wav, sr = sf.read(f_path, dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != TARGET_SR:
            import librosa
            wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)

        noisy = torch.from_numpy(wav).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        enhanced = infer_model(noisy).squeeze(0).squeeze(0).cpu().numpy()

        out_name = os.path.splitext(os.path.basename(f_path))[0] + "_enhanced.wav"
        sf.write(os.path.join(OUTPUT_DIR, out_name), enhanced, TARGET_SR)

print("Xong. File da tang cuong nam trong:", OUTPUT_DIR)
